# 01 — Slide inspection and physical magnification

RocqiPath treats `20x`, `40x`, and `80x` as physical objective
magnifications. A scanner pyramid level is not a magnification: level 1
can represent different physical zooms on different scanners.

This notebook first creates a small ordinary TIFF so every read can be
tested without private data. Change `USE_SYNTHETIC_DEMO` to `False` and
point `SLIDE_PATH` to an SVS/NDPI/TIFF/OME-TIFF for real inspection.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw

from rocqipath.core import SlideReader

USE_SYNTHETIC_DEMO = True
SLIDE_PATH = DATA_ROOT / "wsi" / "example_he.svs"
TARGET_MAGNIFICATION = 20.0

# Required only when the slide has no objective metadata.
SOURCE_MAGNIFICATION_FALLBACK = 40.0

demo_root = Path(
    os.environ.get(
        "ROCQIPATH_NOTEBOOK_DEMO_DIR",
        str(PROJECT_ROOT / "notebook_demo_outputs"),
    )
)
demo_root.mkdir(parents=True, exist_ok=True)


In [ ]:
def create_demo_slide(path: Path, width: int = 2048, height: int = 1536) -> Path:
    '''Create a deterministic tissue-like RGB TIFF for reader testing.'''
    rng = np.random.default_rng(42)
    rgb = np.full((height, width, 3), 246, dtype=np.uint8)
    image = Image.fromarray(rgb)
    draw = ImageDraw.Draw(image)
    draw.rounded_rectangle(
        (180, 160, width - 180, height - 160),
        radius=180,
        fill=(210, 148, 178),
    )
    for _ in range(500):
        x = int(rng.integers(220, width - 220))
        y = int(rng.integers(200, height - 200))
        r = int(rng.integers(4, 13))
        color = (
            int(rng.integers(70, 125)),
            int(rng.integers(40, 90)),
            int(rng.integers(115, 175)),
        )
        draw.ellipse((x - r, y - r, x + r, y + r), fill=color)
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path, format="TIFF")
    image.close()
    return path


if USE_SYNTHETIC_DEMO:
    SLIDE_PATH = create_demo_slide(demo_root / "slide_reader_demo.tif")

if not SLIDE_PATH.is_file():
    raise FileNotFoundError(f"Slide not found: {SLIDE_PATH}")

print(SLIDE_PATH)


## Inspect metadata and resolve the read plan

OpenSlide-backed WSIs expose scanner metadata and native pyramid
downsample factors. Ordinary TIFFs use the Pillow fallback and therefore
need `SOURCE_MAGNIFICATION_FALLBACK`.


In [ ]:
with SlideReader(str(SLIDE_PATH)) as slide:
    print(f"Level-0 dimensions : {slide.dimensions}")
    print(f"Level downsamples  : {slide.level_downsamples}")
    print(f"Metadata fields    : {len(slide.properties)}")

    plan = slide.configure_magnification(
        TARGET_MAGNIFICATION,
        source_magnification=SOURCE_MAGNIFICATION_FALLBACK,
    )
    print("\nResolved magnification plan")
    print(f"  Base objective       : {plan.base_magnification:g}x")
    print(f"  Target objective     : {plan.target_magnification:g}x")
    print(f"  Native pyramid level : {plan.level}")
    print(f"  Native magnification : {plan.native_magnification:g}x")
    print(f"  Final resize factor  : {plan.resize_factor:.4f}")
    print(f"  Target dimensions    : {slide.target_dimensions}")


## Read a center region in target-grid coordinates

After configuration, both `location=(x, y)` and `size=(width, height)`
are expressed in pixels at `TARGET_MAGNIFICATION`. RocqiPath maps the
location to level 0, chooses a native pyramid level, reads enough pixels,
and resizes once to the exact requested zoom.


In [ ]:
PATCH_SIZE = 512

with SlideReader(str(SLIDE_PATH)) as slide:
    plan = slide.configure_magnification(
        TARGET_MAGNIFICATION,
        source_magnification=SOURCE_MAGNIFICATION_FALLBACK,
    )
    target_w, target_h = slide.target_dimensions
    x = max(0, (target_w - PATCH_SIZE) // 2)
    y = max(0, (target_h - PATCH_SIZE) // 2)
    width = min(PATCH_SIZE, target_w)
    height = min(PATCH_SIZE, target_h)
    roi = slide.read_at_magnification((x, y), (width, height)).convert("RGB")

print(f"Target-grid location: {(x, y)}")
print(f"Level-0 location    : {plan.target_to_level0((x, y))}")
print(f"Returned size       : {roi.size}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(roi)
ax.set_title(f"{SLIDE_PATH.name} — center ROI at {TARGET_MAGNIFICATION:g}x")
ax.axis("off")
plt.tight_layout()
plt.show()
roi.close()


In [ ]:
from rocqipath.core import build_magnification_plan

examples = [
    (80.0, 20.0, (1.0, 2.0, 4.0, 8.0)),
    (40.0, 20.0, (1.0, 4.0, 16.0)),
    (20.0, 20.0, (1.0, 2.0, 4.0)),
]

for base, target, downsamples in examples:
    p = build_magnification_plan(base, target, downsamples)
    print(
        f"{base:>4g}x source -> {target:>4g}x output | "
        f"level={p.level}, native={p.native_magnification:g}x, "
        f"resize={p.resize_factor:.3f}"
    )


## Common failures

- **Objective magnification missing**: supply a scanner-specific fallback,
  such as `source_magnification=80.0` for an 80x TMA scan.
- **Requested magnification exceeds source**: RocqiPath rejects 40x output
  from a 20x source rather than inventing resolution.
- **OpenSlide cannot open the format**: install the native OpenSlide runtime
  and confirm the format is supported; ordinary TIFFs can fall back to PIL.
- **libvips error**: slide reading and libvips export are separate native
  dependencies; install both when running extraction or aligned WSI export.

Continue with notebook **02** for tissue-region and TMA extraction.
